# Notebook Custom Video Analyzer Schema

This notebook introduces the **otebiig CustomAnalyzer Schema**. We will be building a custom video analyzer by defining and utilizing a flexible schema, enabling tailored video analysis workflows.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
VIDEO_CONTENT_UNDERSTANDING_KEY=os.getenv('VIDEO_CONTENT_UNDERSTANDING_KEY')

AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
AZURE_AI_API_VERSION = os.getenv("AZURE_AI_API_VERSION", "2025-05-01-preview")
AZURE_AI_API_KEY = os.getenv('AZURE_AI_API_KEY')

ANALYZER_SAMPLE_FILE = '../data/mcp.mp4'
INDEX_NAME = "video-custom-content-understanding-index" 
SEARCH_KEY = os.getenv('SEARCH_KEY')
SEARCH_ENDPOINT = os.getenv('SEARCH_ENDPOINT')

VIDEO_CONTENT_UNDERSTANDING_KEY=os.getenv('VIDEO_CONTENT_UNDERSTANDING_KEY')
AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_EMBEDDING_ENDPOINT=os.getenv('AZURE_OPENAI_EMBEDDING_ENDPOINT')
AZURE_EMBEDDING_DEPLOYMENT_NAME=os.getenv('AZURE_EMBEDDING_DEPLOYMENT_NAME')

AZURE_CHAT_DEPLOYMENT_NAME=os.getenv('AZURE_CHAT_DEPLOYMENT_NAME')
AZURE_OPENAI_LLM_ENDPOINT=os.getenv('AZURE_OPENAI_LLM_ENDPOINT')

AZURE_ENDPOINT=os.getenv('AZURE_ENDPOINT')
AZURE_OPENAI_KEY=os.getenv('AZURE_OPENAI_KEY')


In [2]:
# Defining the schema:

request_body = {
	"description": "custom video analyzer",
	"baseAnalyzerId": "prebuilt-videoAnalyzer",
	"config": {
		"locales": ["en-US"],
		"returnDetails": True,
		"enableFace": False,
		"disableFaceBlurring": False,
		"personDirectoryId": None,
		"segmentationMode": "noSegmentation",
		# "segmentationMode": None,
		"disableContentFiltering": False
	},
	"fieldSchema": {
	"fields": {
		"title": {
			"method": "generate",
			"type": "string",
			"description": "A GenZ-style title for the whole video"
		},
		"summary": {
			"method": "generate",
			"type": "string",
			"description": "A short 4-sentence summary of the full video"
		},
		"motive": {
			"method": "classify",
			"type": "string",
			"enum": ["entertainment", "learning"],
		}
		}
	}
}

In [3]:
import requests
import json

# Replace these with your actual values
endpoint = AZURE_AI_ENDPOINT

# the below analyzer id is a custom id that we need to define.
analyzer_id = "mycustom-videoanalyzer-3"
subscription_key = VIDEO_CONTENT_UNDERSTANDING_KEY

# Construct the URL
url = f"{endpoint}/contentunderstanding/analyzers/{analyzer_id}?api-version=2025-05-01-preview"

# Set the headers
headers = {
    "Ocp-Apim-Subscription-Key": subscription_key,
    "Content-Type": "application/json"
}

# Send the PUT request
response = requests.put(url, headers=headers, json=request_body)

# Print the response
print(f"Status Code: {response.status_code}")
print("Response Headers:", response.headers)
print("Response Body:", response.text)


Status Code: 409
Response Headers: {'Content-Length': '236', 'Content-Type': 'application/json; charset=utf-8', 'request-id': '48e20d7d-8dbb-4e07-b6a9-e8c9a0311c5c', 'x-ms-request-id': '48e20d7d-8dbb-4e07-b6a9-e8c9a0311c5c', 'x-ms-error-code': 'Conflict', 'api-supported-versions': '2025-05-01-preview', 'x-envoy-upstream-service-time': '73', 'apim-request-id': '48e20d7d-8dbb-4e07-b6a9-e8c9a0311c5c', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'Australia East', 'Date': 'Mon, 30 Jun 2025 07:04:27 GMT'}
Response Body: {"error":{"code":"Conflict","message":"The request could not be completed due to a conflict with the current state of the target resource.","innererror":{"code":"ModelExists","message":"A model with the provided name already exists."}}}


In [4]:
import json
import logging
import sys
import time
from collections.abc import Callable
from pathlib import Path
from typing import Any, cast
from dataclasses import dataclass

import requests


def upload_video_to_analyzer():
    settings = Settings(
        endpoint="https://ai-rajanjasani2002-3965.services.ai.azure.com/",
        api_version="2025-05-01-preview",
        # Either subscription_key or aad_token must be provided. Subscription Key is more prioritized.
        subscription_key=VIDEO_CONTENT_UNDERSTANDING_KEY,
        aad_token="AZURE_CONTENT_UNDERSTANDING_AAD_TOKEN",
        # Insert the analyzer name.
        analyzer_id=f"mycustom-videoanalyzer-3",
        # Insert the supported file types of the analyzer.
        file_location="E:/Azure Video Indexer/data/mcp.mp4",
    )
    client = AzureContentUnderstandingClient(
        settings.endpoint,
        settings.api_version,
        subscription_key=settings.subscription_key,
        token_provider=settings.token_provider,
    )
    response = client.begin_analyze(settings.analyzer_id, settings.file_location)
    result = client.poll_result(
        response,
        timeout_seconds=60 * 60,
        polling_interval_seconds=1,
    )
    json.dump(result, sys.stdout, indent=2)
    return result['id']

@dataclass(frozen=True, kw_only=True)
class Settings:
    endpoint: str
    api_version: str
    subscription_key: str | None = None
    aad_token: str | None = None
    analyzer_id: str
    file_location: str

    def __post_init__(self):
        key_not_provided = (
            not self.subscription_key
            or self.subscription_key == "AZURE_CONTENT_UNDERSTANDING_SUBSCRIPTION_KEY"
        )
        token_not_provided = (
            not self.aad_token
            or self.aad_token == "AZURE_CONTENT_UNDERSTANDING_AAD_TOKEN"
        )
        if key_not_provided and token_not_provided:
            raise ValueError(
                "Either 'subscription_key' or 'aad_token' must be provided"
            )

    @property
    def token_provider(self) -> Callable[[], str] | None:
        aad_token = self.aad_token
        if aad_token is None:
            return None

        return lambda: aad_token


class AzureContentUnderstandingClient:
    def __init__(
        self,
        endpoint: str,
        api_version: str,
        subscription_key: str | None = None,
        token_provider: Callable[[], str] | None = None,
        x_ms_useragent: str = "cu-sample-code",
    ) -> None:
        if not subscription_key and token_provider is None:
            raise ValueError(
                "Either subscription key or token provider must be provided"
            )
        if not api_version:
            raise ValueError("API version must be provided")
        if not endpoint:
            raise ValueError("Endpoint must be provided")

        self._endpoint: str = endpoint.rstrip("/")
        self._api_version: str = api_version
        self._logger: logging.Logger = logging.getLogger(__name__)
        self._logger.setLevel(logging.INFO)
        self._headers: dict[str, str] = self._get_headers(
            subscription_key, token_provider and token_provider(), x_ms_useragent
        )

    def begin_analyze(self, analyzer_id: str, file_location: str):
        """
        Begins the analysis of a file or URL using the specified analyzer.

        Args:
            analyzer_id (str): The ID of the analyzer to use.
            file_location (str): The path to the file or the URL to analyze.

        Returns:
            Response: The response from the analysis request.

        Raises:
            ValueError: If the file location is not a valid path or URL.
            HTTPError: If the HTTP request returned an unsuccessful status code.
        """
        if Path(file_location).exists():
            with open(file_location, "rb") as file:
                data = file.read()
            headers = {"Content-Type": "application/octet-stream"}
        elif "https://" in file_location or "http://" in file_location:
            data = {"url": file_location}
            headers = {"Content-Type": "application/json"}
        else:
            raise ValueError("File location must be a valid path or URL.")

        headers.update(self._headers)
        if isinstance(data, dict):
            response = requests.post(
                url=self._get_analyze_url(
                    self._endpoint, self._api_version, analyzer_id
                ),
                headers=headers,
                json=data,
            )
        else:
            response = requests.post(
                url=self._get_analyze_url(
                    self._endpoint, self._api_version, analyzer_id
                ),
                headers=headers,
                data=data,
            )

        response.raise_for_status()
        self._logger.info(
            f"Analyzing file {file_location} with analyzer: {analyzer_id}"
        )
        return response

    def poll_result(
        self,
        response: requests.Response,
        timeout_seconds: int = 120,
        polling_interval_seconds: int = 2,
    ) -> dict[str, Any]:  # pyright: ignore[reportExplicitAny]
        """
        Polls the result of an asynchronous operation until it completes or times out.

        Args:
            response (Response): The initial response object containing the operation location.
            timeout_seconds (int, optional): The maximum number of seconds to wait for the operation to complete. Defaults to 120.
            polling_interval_seconds (int, optional): The number of seconds to wait between polling attempts. Defaults to 2.

        Raises:
            ValueError: If the operation location is not found in the response headers.
            TimeoutError: If the operation does not complete within the specified timeout.
            RuntimeError: If the operation fails.

        Returns:
            dict: The JSON response of the completed operation if it succeeds.
        """
        operation_location = response.headers.get("operation-location", "")
        if not operation_location:
            raise ValueError("Operation location not found in response headers.")

        headers = {"Content-Type": "application/json"}
        headers.update(self._headers)

        start_time = time.time()
        while True:
            elapsed_time = time.time() - start_time
            self._logger.info(
                "Waiting for service response", extra={"elapsed": elapsed_time}
            )
            if elapsed_time > timeout_seconds:
                raise TimeoutError(
                    f"Operation timed out after {timeout_seconds:.2f} seconds."
                )

            response = requests.get(operation_location, headers=self._headers)
            response.raise_for_status()
            result = cast(dict[str, str], response.json())
            status = result.get("status", "").lower()
            if status == "succeeded":
                self._logger.info(
                    f"Request result is ready after {elapsed_time:.2f} seconds."
                )
                return response.json()  # pyright: ignore[reportAny]
            elif status == "failed":
                self._logger.error(f"Request failed. Reason: {response.json()}")
                raise RuntimeError("Request failed.")
            else:
                self._logger.info(
                    f"Request {operation_location.split('/')[-1].split('?')[0]} in progress ..."
                )
            time.sleep(polling_interval_seconds)

    def _get_analyze_url(self, endpoint: str, api_version: str, analyzer_id: str):
        return f"{endpoint}/contentunderstanding/analyzers/{analyzer_id}:analyze?api-version={api_version}&stringEncoding=utf16"

    def _get_headers(
        self, subscription_key: str | None, api_token: str | None, x_ms_useragent: str
    ) -> dict[str, str]:
        """Returns the headers for the HTTP requests.
        Args:
            subscription_key (str): The subscription key for the service.
            api_token (str): The API token for the service.
            enable_face_identification (bool): A flag to enable face identification.
        Returns:
            dict: A dictionary containing the headers for the HTTP requests.
        """
        headers = (
            {"Ocp-Apim-Subscription-Key": subscription_key}
            if subscription_key
            else {"Authorization": f"Bearer {api_token}"}
        )
        headers["x-ms-useragent"] = x_ms_useragent
        return headers


if __name__ == "__main__":
    video_id = upload_video_to_analyzer()


{
  "id": "02cd548b-cd37-4627-860f-72dc60c08436",
  "status": "Succeeded",
  "result": {
    "analyzerId": "mycustom-videoanalyzer-3",
    "apiVersion": "2025-05-01-preview",
    "createdAt": "2025-06-30T07:04:33Z",
    "stringEncoding": "utf8",
    "warnings": [],
    "contents": [
      {
        "markdown": "# Video: 00:00.000 => 03:45.792\nWidth: 1280\nHeight: 720\n\nTranscript\n```\nWEBVTT\n\n00:00.320 --> 00:04.800\n<Speaker 1>If you're building AI agents, you probably heard about MCP, our Model Context Protocol.\n\n00:05.200 --> 00:11.160\n<Speaker 1>MCP is a new open source standard to connect your agents to data sources such as databases or APIs.\n\n00:11.600 --> 00:13.760\n<Speaker 1>MCP consists of multiple components.\n\n00:13.920 --> 00:17.120\n<Speaker 1>The most important ones are the host, the client, and the server.\n\n00:17.200 --> 00:18.160\n<Speaker 1>So let's break it down.\n\n00:19.560 --> 00:22.080\n<Speaker 1>At the very top, you would have your MCP host.\n\n00:

In [5]:
import requests

def get_analyzer_result(result_id):
    url = f"{AZURE_AI_ENDPOINT}/contentunderstanding/analyzerResults/{result_id}?api-version=2025-05-01-preview"
    headers = {
        "Ocp-Apim-Subscription-Key": VIDEO_CONTENT_UNDERSTANDING_KEY
    }

    response = requests.get(url, headers=headers)

    print(f"Status Code: {response.status_code}")
    print(f"Response Headers: {response.headers}")
    print(f"Response Body: {response.text}")

    return response


In [6]:
index_name = INDEX_NAME
'''
No need to run this cell again.
This code block was only used to create the Azure Cognitive Search index.
It has been commented out to prevent accidental re-execution.
'''

from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes.models import (
    SearchField,
    SimpleField,
    SearchableField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmMetric,
    ExhaustiveKnnAlgorithmConfiguration,
    ExhaustiveKnnParameters,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    SearchIndex,
    AzureOpenAIParameters,
)
admin_key = SEARCH_KEY
index_name = INDEX_NAME

client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=AzureKeyCredential(admin_key)
)

# Define the index fields
fields = [
    SimpleField(name="videoId", type="Edm.String", key=True),
    SearchableField(name="title", type="Edm.String"),
    SearchableField(name="transcript", type="Edm.String", vector_search_embedding_configuration="myOpenAI"),

    SearchableField(name="summary", type="Edm.String", vector_search_embedding_configuration="myOpenAI"),
    SearchableField(name="motive", type="Edm.String"),
    SearchField(
        name="summaryVector", 
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single), vector_search_dimensions=1536, vector_search_profile_name="myHnswProfile"
    ),
    SearchField(
        name="transcriptVector", 
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single), vector_search_dimensions=1536, vector_search_profile_name="myHnswProfile"
    ),
]

# Vectore search profile 
vector_search = VectorSearch(  
    algorithms=[  
        HnswAlgorithmConfiguration(  
            name="myHnsw",  
            parameters=HnswParameters(  
                m=4,  
                ef_construction=400,  
                ef_search=500,  
                metric=VectorSearchAlgorithmMetric.COSINE,  
            ),  
        ),  
        ExhaustiveKnnAlgorithmConfiguration(  
            name="myExhaustiveKnn",  
            parameters=ExhaustiveKnnParameters(  
                metric=VectorSearchAlgorithmMetric.COSINE,  
            ),  
        ),  
    ],  
    profiles=[  
        VectorSearchProfile(  
            name="myHnswProfile",  
            algorithm_configuration_name="myHnsw",  
            vectorizer="myOpenAI",  
        )
    ],  
    vectorizers=[  
        AzureOpenAIVectorizer(  
            name="myOpenAI",  
            kind="azureOpenAI",  
            azure_open_ai_parameters=AzureOpenAIParameters(  
                resource_uri=AZURE_ENDPOINT,  
                deployment_id="text-embedding-ada-002dev",  
                api_key=AZURE_OPENAI_KEY,  
            ),  
        ),  
    ],
)  

# Create the search index
index = SearchIndex(
    name=index_name,  # Make sure index_name is defined
    fields=fields,
    vector_search=vector_search,
)

# Create the index/
try:
    result = client.create_or_update_index(index)
    print(f"Index '{index_name}' created successfully")
except Exception as e:
    print(f"Error creating index: {e}")

Index 'video-custom-content-understanding-index' created successfully


In [7]:
get_analyzer_result(video_id)

Status Code: 200
Response Headers: {'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json', 'request-id': 'aa5bdeee-b4fd-42ec-95f8-faaca9ce543b', 'x-ms-request-id': 'aa5bdeee-b4fd-42ec-95f8-faaca9ce543b', 'api-supported-versions': '2024-12-01-preview,2025-05-01-preview', 'x-envoy-upstream-service-time': '41', 'apim-request-id': 'aa5bdeee-b4fd-42ec-95f8-faaca9ce543b', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'Australia East', 'Date': 'Mon, 30 Jun 2025 07:05:32 GMT'}
Response Body: {"id":"02cd548b-cd37-4627-860f-72dc60c08436","status":"Succeeded","result":{"analyzerId":"mycustom-videoanalyzer-3","apiVersion":"2025-05-01-preview","createdAt":"2025-06-30T07:04:33Z","stringEncoding":"utf8","warnings":[],"contents":[{"markdown":"# Video: 00:00.000 => 03:45.792\nWidth: 1280\nHeight: 720\n\nTranscript\n```\nWEBVTT\n\n00:00.320 --> 00:04.800\n<Speaker 1>If you're building AI agents, you probably 

<Response [200]>

In [8]:
import json
import uuid
from openai import AzureOpenAI
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

Azure_client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2023-05-15",  # Use your deployed API version
    azure_endpoint=AZURE_OPENAI_EMBEDDING_ENDPOINT  # e.g., "https://<your-resource-name>.openai.azure.com/"
)

def create_embedding(text: str) -> list[float]:
    response = Azure_client.embeddings.create(
        input=text,
        model=AZURE_EMBEDDING_DEPLOYMENT_NAME
    )
    return response.data[0].embedding

response = get_analyzer_result(video_id)
response = response.json()
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=index_name,
    credential=AzureKeyCredential(admin_key)
)

title = response['result']['contents'][0]['fields']['title']['valueString']
summary = response['result']['contents'][0]['fields']['summary']['valueString']
motive = response['result']['contents'][0]['fields']['motive']['valueString']
phrases = response['result']['contents'][0].get('transcriptPhrases', [])
transcript_text = " ".join([phrase['text'] for phrase in phrases])

doc = { 
    'videoId': video_id,
    'title': title,
    'summary': summary,
    'motive': motive,
    'transcript': transcript_text,
    'transcriptVector' : create_embedding(transcript_text) ,
    'summaryVector' : create_embedding(transcript_text) ,
}

search_client.upload_documents(documents=[doc])

Status Code: 200
Response Headers: {'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json', 'request-id': 'c6df3d25-4547-4443-a4da-f6fb30dffa1e', 'x-ms-request-id': 'c6df3d25-4547-4443-a4da-f6fb30dffa1e', 'api-supported-versions': '2024-12-01-preview,2025-05-01-preview', 'x-envoy-upstream-service-time': '34', 'apim-request-id': 'c6df3d25-4547-4443-a4da-f6fb30dffa1e', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'Australia East', 'Date': 'Mon, 30 Jun 2025 07:05:37 GMT'}
Response Body: {"id":"02cd548b-cd37-4627-860f-72dc60c08436","status":"Succeeded","result":{"analyzerId":"mycustom-videoanalyzer-3","apiVersion":"2025-05-01-preview","createdAt":"2025-06-30T07:04:33Z","stringEncoding":"utf8","warnings":[],"contents":[{"markdown":"# Video: 00:00.000 => 03:45.792\nWidth: 1280\nHeight: 720\n\nTranscript\n```\nWEBVTT\n\n00:00.320 --> 00:04.800\n<Speaker 1>If you're building AI agents, you probably 

In [9]:
def IndexSearch(question_vector):
    """
    Function information:
        Info: Do vector search from exechat index  
    """
    try:
        
        # Headers and Params value 
        headers = {'Content-Type': 'application/json', 'api-key': SEARCH_KEY }
        params = {'api-version': '2023-11-01'}

        index_payload = {
            "select": "title,motive,transcript,summary",  # Combine all fields you want to retrieve
            "count": True,
            "vectorQueries": [
                {
                    "vector": question_vector,
                    "fields": "transcriptVector",
                    "k": 3,
                    "kind": "vector"
                },
                {
                    "vector": question_vector,
                    "fields": "summaryVector",
                    "k": 3,
                    "kind": "vector"
                }
            ]
        }


        Request = requests.post(SEARCH_ENDPOINT + "/indexes/" + INDEX_NAME +"/docs/search", 
            data=json.dumps(index_payload), headers=headers, params=params)
        Response = Request.json()
        return True, Response['value']
    except Exception as e:
        print(e)
        return False, None

In [ ]:
from openai import AzureOpenAI

def ask_question_with_context(user_question):
    user_question_vector = create_embedding(user_question)
    context = IndexSearch(user_question_vector)

    # Azure OpenAI client
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version="2024-12-01-preview",
        azure_endpoint=AZURE_OPENAI_LLM_ENDPOINT
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # This must be your deployed name (e.g., "gpt-4-deployment")
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Use ONLY the provided context to answer in detail the user's question. "
                    "Do not rely on any outside knowledge or make assumptions beyond the context. "
                    "If the answer cannot be found in the context, reply with: 'The context does not contain that information.'"
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}"
            },
            {
                "role": "user",
                "content": f"Question: {user_question}"
            }
        ],
        temperature=0.3,
        max_tokens=1024
    )

    return response.choices[0].message.content


In [11]:
ask_question_with_context("What is MCP?")

(True, [{'@search.score': 0.03333333507180214, 'title': 'MCP Unlocked: The Future of AI Connectivity', 'transcript': "If you're building AI agents, you've probably heard about MCP, our Model Context Protocol. MCP is a new open-source standard to connect your agents to data sources such as databases or APIs. MCP consists of multiple components. The most important ones are the host, the client, and the server. So let's break it down. At the very top, you would have your MCP host. Your MCP host will include an MCP client, and it could also include multiple clients. The MCP host could be an application such as a chat app. It could also be a code assistant in your IDE and much more. The MCP host will connect to an MCP server. It can actually connect to multiple MCP servers as well. It doesn't matter how many MCP servers you connect to your MCP host or client. The MCP hosts and servers will connect over each other through the MCP protocol. The MCP protocol is a transport layer in the middle.

"MCP, or Model Context Protocol, is a new open-source standard designed to connect AI agents to various data sources such as databases or APIs. It consists of multiple components, with the most important being the host, client, and server. \n\nThe MCP host can be an application like a chat app or a code assistant in an Integrated Development Environment (IDE). The host connects to one or more MCP servers through the MCP protocol, which serves as a transport layer. When the MCP host or client needs a tool, it communicates with the MCP server, which can connect to different types of data sources, including relational databases, NoSQL databases, APIs, and local files.\n\nIn practice, when a user interacts with an MCP-enabled application (like a chat app), the MCP host retrieves tools from the MCP server based on the user's query. The server determines which tools are available, and the host then connects to a large language model to process the question alongside the available tools. The 